# Data Understanding - 06/07/2026

## Proyecto StaySpain

**Dataset de origen:** `raw_dataset_29_06_2026.csv`  

Este notebook proporciona la evidencia reproducible que respalda el documento
`Data_Understanding_06_07_2026.pdf`.

### Objetivo

Verificar si el dataset contiene la información necesaria para responder a las
preguntas de negocio del Sprint 2, identificar riesgos de calidad y posibles
sesgos, y definir las comprobaciones necesarias para las fases posteriores.

### Alcance

Este notebook se limita a **Data Understanding**:

1. dimensiones, estructura y tipos de datos;
2. unidad de análisis y snapshots;
3. cobertura y valores faltantes;
4. variables necesarias para cada perfil;
5. riesgos iniciales de calidad, representatividad y sesgo;
6. adecuación del dataset para el Sprint 2.

> **Fuera de alcance:** correlaciones, pruebas de hipótesis, tratamiento
> definitivo de outliers, comparación Best/Worst y conclusiones de negocio.

## Preguntas de negocio del Sprint 2

### Marketing y Estrategia Comercial

**¿Qué características de los alojamientos -equipamientos, capacidad y
ubicación- están más relacionadas con los precios en cada ciudad?**

### Operaciones y Gestión de Inventario

**¿Qué impacto tiene la opción de reserva instantánea, sin revisión del
propietario, sobre la disponibilidad media en cada ciudad?**

### Experiencia del Cliente

**¿Qué aspectos -precisión de los detalles, limpieza, check-in o comunicación-
presentan las mayores diferencias entre los alojamientos mejor y peor
valorados en la evaluación general?**

## 1. Librerías y configuración visual

La presentación sigue la paleta acordada para StaySpain:

- **azul oscuro:** encabezados y estructura principal;
- **azul claro:** filas alternas y elementos destacados;
- **gris:** información contextual y filas secundarias.

In [1]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

NAVY = "#164B6B"
HEADER_BLUE = "#1F6D9B"
ROW_BLUE = "#B9D9EA"
ROW_GREY = "#E7EDF1"
ROW_HOVER = "#9CCBE3"
TEXT_DARK = "#172B36"
BORDER_BLUE = "#6D9EBB"
WHITE = "#FFFFFF"

### 1.1 Funciones auxiliares

In [2]:
def require_columns(dataframe, columns):
    """Comprueba que existen todas las columnas necesarias."""
    missing_columns = [
        column
        for column in columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            "Faltan columnas necesarias: "
            f"{missing_columns}"
        )


def latest_snapshot(dataframe):
    """Conserva el último snapshot de cada alojamiento."""
    ordered = dataframe.sort_values(
        ["apartment_id", "insert_date_parsed"]
    )

    latest = ordered.drop_duplicates(
        subset="apartment_id",
        keep="last",
    )

    return latest.copy()

In [3]:
def coverage_by_group(
    dataframe,
    group_column,
    value_column,
):
    """Calcula la cobertura válida de una variable por grupo."""
    coverage = (
        dataframe.groupby(group_column)
        .agg(
            total=("apartment_id", "size"),
            valid=(value_column, "count"),
        )
        .reset_index()
    )

    coverage["coverage_pct"] = (
        coverage["valid"]
        / coverage["total"]
        * 100
    )

    return coverage


def count_listing_changes(
    dataframe,
    variables,
):
    """Cuenta alojamientos repetidos con cambios por variable."""
    rows = []

    for variable in variables:
        changed = (
            dataframe.groupby("apartment_id")[variable]
            .nunique(dropna=False)
            .gt(1)
            .sum()
        )

        rows.append(
            {
                "Variable": variable,
                "Alojamientos con cambios": int(changed),
            }
        )

    return pd.DataFrame(rows)

In [4]:
def show_table(dataframe, caption, formats=None):
    """Muestra una tabla legible con el estilo visual de StaySpain."""
    formats = formats or {}

    table_styles = [
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),("font-size", "16px"),("font-weight", "700"),("color", HEADER_BLUE),
                ("padding", "10px 4px"),("text-align", "left"),
            ],
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),("font-size", "14px"),("min-width", "520px"),
                ("max-width", "950px"),("background-color", WHITE),("border", f"1px solid {BORDER_BLUE}"),
            ],
        },
        {
            "selector": "thead th",
            "props": [
                ("background-color", NAVY),("color", WHITE),("font-weight", "700"),
                ("text-align", "center"),("padding", "9px 12px"),("border", f"1px solid {WHITE}"),
            ],
        },
        {
            "selector": "tbody td",
            "props": [
                ("color", TEXT_DARK),("font-weight", "500"),("padding", "8px 12px"),("border", f"1px solid {WHITE}"),
            ],
        },
        {
            "selector": "tbody tr:nth-child(odd)",
            "props": [
                ("background-color", ROW_BLUE),
            ],
        },
        {
            "selector": "tbody tr:nth-child(even)",
            "props": [
                ("background-color", ROW_GREY),
            ],
        },
        {
            "selector": "tbody tr:hover",
            "props": [
                ("background-color", ROW_HOVER),
            ],
        },
        {
            "selector": "tbody tr:hover td",
            "props": [
                ("color", TEXT_DARK),
            ],
        },
    ]

    styled = (
        dataframe.style
        .hide(axis="index")
        .format(formats, na_rep="—")
        .set_caption(caption)
        .set_table_styles(table_styles)
    )

    first_column = dataframe.columns[0]

    styled = styled.set_properties(
        subset=[first_column],
        **{
            "font-weight": "700",
            "color": TEXT_DARK,
            "text-align": "left",
        },
    )

    numeric_columns = (
        dataframe
        .select_dtypes(include="number")
        .columns
        .tolist()
    )

    if numeric_columns:
        styled = styled.set_properties(
            subset=numeric_columns,
            **{
                "text-align": "right",
                "color": TEXT_DARK,
            },
        )

    display(styled)

## 2. Carga y validación mínima del dataset

Se utiliza únicamente la ruta relativa acordada para el repositorio. El
notebook no muestra rutas personales ni información del sistema local.

In [5]:
df = pd.read_csv( "../Data/raw_dataset_29_06_2026.csv")

df["city"] = (
    df["city"]
    .str.strip()
    .str.capitalize()
)

print("Dataset cargado correctamente.")
print(
    f"Dimensiones: {df.shape[0]:,} filas "
    f"y {df.shape[1]} columnas."
)

Dataset cargado correctamente.
Dimensiones: 7,001 filas y 35 columnas.


In [6]:
required_columns = [
    "apartment_id","city","insert_date","price","amenities_list","is_instant_bookable",
    "availability_30","availability_60","availability_90","availability_365","review_scores_rating",
    "review_scores_accuracy","review_scores_cleanliness","review_scores_checkin","review_scores_communication",
]

require_columns(
    dataframe=df,
    columns=required_columns,
)

print("Validación de columnas principales completada.")

Validación de columnas principales completada.


## 3. Dimensiones y estructura general

In [7]:
dimension_summary = pd.DataFrame(
    {
        "Indicador": [
            "Total de registros","Total de variables","Alojamientos únicos","Registros adicionales",
            "IDs con más de una observación","Duplicados exactos","Ciudades representadas",
        ],
        "Valor": [
            len(df),
            df.shape[1],
            df["apartment_id"].nunique(),
            len(df) - df["apartment_id"].nunique(),
            df["apartment_id"].value_counts().gt(1).sum(),
            df.duplicated().sum(),
            df["city"].nunique(),
        ],
    }
)

show_table(
    dimension_summary,
    "Dimensiones principales del dataset",
    formats={"Valor": "{:,.0f}"},
)

Indicador,Valor
Total de registros,"7,001"
Total de variables,35
Alojamientos únicos,"6,733"
Registros adicionales,268
IDs con más de una observación,261
Duplicados exactos,0
Ciudades representadas,8


In [8]:
schema_summary = pd.DataFrame(
    {
        "Campo": df.columns,
        "Tipo observado": df.dtypes.astype(str).values,
        "Nulos": df.isna().sum().values,
        "% nulos": (
            df.isna().mean().values
            * 100
        ),
    }
)

show_table(
    schema_summary,
    "Estructura, tipos y cobertura por variable",
    formats={
        "Nulos": "{:,.0f}",
        "% nulos": "{:.2f}%",
    },
)

Campo,Tipo observado,Nulos,% nulos
apartment_id,int64,0,0.00%
name,str,3,0.04%
description,str,29,0.41%
host_id,int64,0,0.00%
neighbourhood_name,str,0,0.00%
neighbourhood_district,str,"2,760",39.42%
room_type,str,0,0.00%
accommodates,int64,0,0.00%
bathrooms,float64,32,0.46%
bedrooms,float64,29,0.41%


### Interpretación

El dataset contiene **7.001 registros y 35 variables**. Se identifican
**6.733 alojamientos únicos**, 268 registros adicionales y ningún duplicado
exacto. Los registros repetidos deben analizarse como posibles snapshots.

## 4. Unidad de análisis y dimensión temporal

In [9]:
df_work = df.copy()

df_work["insert_date_parsed"] = pd.to_datetime(
    df_work["insert_date"],
    format="%d/%m/%Y",
    errors="coerce",
)

invalid_dates = (
    df_work["insert_date"].notna()
    & df_work["insert_date_parsed"].isna()
).sum()

print(
    "Fechas de extracción no interpretables: "
    f"{invalid_dates}"
)

Fechas de extracción no interpretables: 0


In [10]:
snapshot_frequency = (
    df_work["apartment_id"]
    .value_counts()
    .value_counts()
    .sort_index()
    .rename_axis("Observaciones por alojamiento")
    .reset_index(name="Número de alojamientos")
)

show_table(
    snapshot_frequency,
    "Frecuencia de observaciones por apartment_id",
    formats={
        "Observaciones por alojamiento": "{:,.0f}",
        "Número de alojamientos": "{:,.0f}",
    },
)

Observaciones por alojamiento,Número de alojamientos
1,"6,472"
2,254
3,7


In [11]:
repeated_ids = (
    df_work["apartment_id"]
    .value_counts()
)

repeated_ids = repeated_ids[
    repeated_ids.gt(1)
].index

repeated_df = df_work[
    df_work["apartment_id"].isin(repeated_ids)
].copy()

snapshot_variables = [
    "price","amenities_list","is_instant_bookable","availability_30","availability_60","availability_90","availability_365",
]

snapshot_changes = count_listing_changes(
    dataframe=repeated_df,
    variables=snapshot_variables,
)

In [12]:
availability_columns = [
    "availability_30","availability_60","availability_90","availability_365",
]

changed_availability = (
    repeated_df
    .groupby("apartment_id")[availability_columns]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
    .sum()
)

availability_row = pd.DataFrame(
    {
        "Variable": [
            "Algún horizonte de disponibilidad"
        ],
        "Alojamientos con cambios": [
            int(changed_availability)
        ],
    }
)

snapshot_changes = pd.concat(
    [
        snapshot_changes,
        availability_row,
    ],
    ignore_index=True,
)

show_table(
    snapshot_changes,
    "Cambios observados entre snapshots",
    formats={
        "Alojamientos con cambios": "{:,.0f}"
    },
)

Variable,Alojamientos con cambios
price,121
amenities_list,158
is_instant_bookable,22
availability_30,201
availability_60,210
availability_90,220
availability_365,235
Algún horizonte de disponibilidad,236


In [13]:
df_latest = latest_snapshot(df_work)

if not df_latest["apartment_id"].is_unique:
    raise ValueError(
        "El último snapshot contiene IDs duplicados."
    )

print(
    "Alojamientos en el último snapshot disponible: "
    f"{len(df_latest):,}"
)

Alojamientos en el último snapshot disponible: 6,733


In [14]:
snapshot_years = (
    df_latest["insert_date_parsed"]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("Año del último snapshot")
    .reset_index(name="Alojamientos")
)

snapshot_years["% del total"] = (
    snapshot_years["Alojamientos"]
    / snapshot_years["Alojamientos"].sum()
    * 100
)

show_table(
    snapshot_years,
    "Distribución temporal del último snapshot",
    formats={
        "Año del último snapshot": "{:.0f}",
        "Alojamientos": "{:,.0f}",
        "% del total": "{:.2f}%",
    },
)

Año del último snapshot,Alojamientos,% del total
2017,773,11.48%
2018,"1,910",28.37%
2019,"2,151",31.95%
2020,"1,631",24.22%
2021,268,3.98%


### Interpretación

Los registros repetidos presentan cambios en variables relevantes para el
Sprint 2. Esto confirma que son snapshots y no duplicados exactos.

Para describir el **último estado disponible en el dataset**, se conserva la
observación más reciente de cada `apartment_id`. Los últimos snapshots se
distribuyen entre 2017 y 2021, por lo que existe un riesgo de no
contemporaneidad. `insert_date` debe mantenerse como variable de control.

## 5. Distribución general por ciudad

In [15]:
city_distribution = (
    df["city"]
    .value_counts()
    .rename_axis("Ciudad")
    .reset_index(name="Registros")
)

city_distribution["% del total"] = (
    city_distribution["Registros"]
    / city_distribution["Registros"].sum()
    * 100
)

show_table(
    city_distribution,
    "Distribución de registros por ciudad",
    formats={
        "Registros": "{:,.0f}",
        "% del total": "{:.2f}%",
    },
)

Ciudad,Registros,% del total
Barcelona,"2,127",30.38%
Madrid,"1,446",20.65%
Mallorca,"1,144",16.34%
Girona,"1,125",16.07%
Sevilla,361,5.16%
Malaga,350,5.00%
Valencia,307,4.39%
Menorca,141,2.01%


### Interpretación

La representación territorial es desigual. Barcelona concentra el mayor
volumen y Menorca el menor. Esta diferencia no demuestra por sí sola la
existencia de sesgo, pero puede afectar a la precisión de las comparaciones.

## 6. Variables necesarias por perfil

In [16]:
role_variables = {
    "Marketing y Estrategia Comercial": [
        "price","amenities_list","accommodates","bathrooms","bedrooms","beds","room_type",
        "city","neighbourhood_name","neighbourhood_district","insert_date",
    ],
    "Operaciones y Gestión de Inventario": [
        "is_instant_bookable","availability_30","availability_60","availability_90",
        "availability_365","city","insert_date",
    ],
    "Experiencia del Cliente": [
        "apartment_id","city","review_scores_rating","review_scores_accuracy","review_scores_cleanliness",
        "review_scores_checkin","review_scores_communication","number_of_reviews","review_scores_location",
        "price","room_type","insert_date",
    ],
}

In [17]:
role_rows = []

for role, variables in role_variables.items():
    missing_columns = [
        column
        for column in variables
        if column not in df.columns
    ]

    role_rows.append(
        {
            "Perfil": role,
            "Variables requeridas": len(variables),
            "Variables disponibles": (
                len(variables)
                - len(missing_columns)
            ),
            "Columnas ausentes": (
                ", ".join(missing_columns)
                if missing_columns
                else "Ninguna"
            ),
            "Adecuación estructural": (
                "Sí"
                if not missing_columns
                else "No"
            ),
        }
    )

role_check = pd.DataFrame(role_rows)

show_table(
    role_check,
    "Disponibilidad de variables por perfil",
    formats={
        "Variables requeridas": "{:,.0f}",
        "Variables disponibles": "{:,.0f}",
    },
)

Perfil,Variables requeridas,Variables disponibles,Columnas ausentes,Adecuación estructural
Marketing y Estrategia Comercial,11,11,Ninguna,Sí
Operaciones y Gestión de Inventario,7,7,Ninguna,Sí
Experiencia del Cliente,12,12,Ninguna,Sí


### Interpretación

El dataset bruto contiene las columnas necesarias para las tres preguntas.
Todavía deben revisarse cobertura, formato, escala y consistencia semántica.

## 7. Valores nulos relevantes

In [18]:
null_columns = [
    "neighbourhood_district","review_scores_checkin","review_scores_accuracy","review_scores_communication",
    "review_scores_cleanliness","review_scores_rating","price","bathrooms","bedrooms","amenities_list",
    "beds","is_instant_bookable","availability_30","availability_60","availability_90","availability_365",
]

null_summary = pd.DataFrame(
    {
        "Campo": null_columns,
        "Nulos": [
            df[column].isna().sum()
            for column in null_columns
        ],
    }
)

null_summary["% nulos"] = (
    null_summary["Nulos"]
    / len(df)
    * 100
)

show_table(
    null_summary,
    "Valores nulos relevantes para el Sprint 2",
    formats={
        "Nulos": "{:,.0f}",
        "% nulos": "{:.2f}%",
    },
)

Campo,Nulos,% nulos
neighbourhood_district,"2,760",39.42%
review_scores_checkin,"1,341",19.15%
review_scores_accuracy,"1,336",19.08%
review_scores_communication,"1,332",19.03%
review_scores_cleanliness,"1,330",19.00%
review_scores_rating,"1,327",18.95%
price,131,1.87%
bathrooms,32,0.46%
bedrooms,29,0.41%
amenities_list,17,0.24%


### 7.1 Cobertura geográfica

In [19]:
district_coverage = (
    df.groupby("city")["neighbourhood_district"]
    .agg(
        Alojamientos="size",
        Nulos=lambda series: (
            series.isna().sum()
        ),
        Distritos_distintos="nunique",
    )
    .reset_index()
)

district_coverage["% nulo"] = (
    district_coverage["Nulos"]
    / district_coverage["Alojamientos"]
    * 100
)

district_coverage = district_coverage.rename(
    columns={"city": "Ciudad"}
)

show_table(
    district_coverage,
    "Cobertura de neighbourhood_district por ciudad",
    formats={
        "Alojamientos": "{:,.0f}",
        "Nulos": "{:,.0f}",
        "% nulo": "{:.2f}%",
        "Distritos_distintos": "{:,.0f}",
    },
)

Ciudad,Alojamientos,Nulos,Distritos_distintos,% nulo
Barcelona,"2,127",0,10,0.00%
Girona,"1,125","1,125",0,100.00%
Madrid,"1,446",0,21,0.00%
Malaga,350,350,0,100.00%
Mallorca,"1,144","1,144",0,100.00%
Menorca,141,141,0,100.00%
Sevilla,361,0,11,0.00%
Valencia,307,0,19,0.00%


In [20]:
neighbourhood_summary = pd.DataFrame(
    {
        "Indicador": [
            "Categorías de neighbourhood_name","Nulos en neighbourhood_name",
        ],
        "Valor": [
            df_latest["neighbourhood_name"].nunique(),
            df_latest["neighbourhood_name"].isna().sum(),
        ],
    }
)

show_table(
    neighbourhood_summary,
    "Cardinalidad de neighbourhood_name",
    formats={"Valor": "{:,.0f}"},
)

Indicador,Valor
Categorías de neighbourhood_name,478
Nulos en neighbourhood_name,0


### Interpretación

`neighbourhood_district` está completo únicamente en Barcelona, Madrid,
Sevilla y Valencia. Es 100 % nulo en Girona, Malaga, Mallorca y Menorca.

`neighbourhood_name` está completo, pero tiene una cardinalidad elevada.

### 7.2 Patrón de valores faltantes en reseñas

In [21]:
review_pattern_columns = [
    "review_scores_rating","first_review_date","last_review_date","reviews_per_month",
]

review_missing_pattern = (
    df_latest[review_pattern_columns]
    .isna()
    .value_counts()
    .reset_index(name="Alojamientos")
)

review_missing_pattern = review_missing_pattern.rename(
    columns={
        column: f"{column} nulo"
        for column in review_pattern_columns
    }
)

show_table(
    review_missing_pattern,
    "Patrones de valores faltantes en variables de reseña",
    formats={"Alojamientos": "{:,.0f}"},
)

review_scores_rating nulo,first_review_date nulo,last_review_date nulo,reviews_per_month nulo,Alojamientos
False,False,False,False,"5,457"
True,True,True,True,"1,202"
True,False,False,False,71
False,False,True,False,2
True,True,False,True,1


In [22]:
no_rating = df_latest[
    df_latest["review_scores_rating"].isna()
].copy()

no_rating_review_history = pd.DataFrame(
    {
        "Indicador": [
            "Alojamientos sin rating","Sin first_review_date","Sin last_review_date",
            "Sin reviews_per_month","Con number_of_reviews igual a 0",
        ],
        "Valor": [
            len(no_rating),
            no_rating["first_review_date"].isna().sum(),
            no_rating["last_review_date"].isna().sum(),
            no_rating["reviews_per_month"].isna().sum(),
            no_rating["number_of_reviews"].eq(0).sum(),
        ],
    }
)

show_table(
    no_rating_review_history,
    "Historial de reseñas en alojamientos sin rating",
    formats={"Valor": "{:,.0f}"},
)

Indicador,Valor
Alojamientos sin rating,"1,274"
Sin first_review_date,"1,203"
Sin last_review_date,"1,202"
Sin reviews_per_month,"1,203"
Con number_of_reviews igual a 0,"1,200"


### Interpretación

La coincidencia entre rating, fechas de reseña y `reviews_per_month` respalda
que la mayoría de los nulos corresponden a alojamientos sin historial de
valoraciones. No deben imputarse las puntuaciones.

### 7.3 Cobertura de valoraciones

In [23]:
experience_columns = [
    "review_scores_rating","review_scores_accuracy","review_scores_cleanliness","review_scores_checkin",
    "review_scores_communication",
]

valid_rating_count = (
    df_latest["review_scores_rating"]
    .notna()
    .sum()
)

complete_experience_count = (
    df_latest[experience_columns]
    .notna()
    .all(axis=1)
    .sum()
)

rating_coverage = (
    valid_rating_count
    / len(df_latest)
    * 100
)

complete_coverage = (
    complete_experience_count
    / len(df_latest)
    * 100
)

In [24]:
coverage_summary = pd.DataFrame(
    {
        "Indicador": [
            "Alojamientos en el último snapshot","Alojamientos con rating válido","Cobertura de rating",
            "Alojamientos con cinco campos válidos","Cobertura completa de Experiencia",
        ],
        "Valor": [
            len(df_latest),
            valid_rating_count,
            rating_coverage,
            complete_experience_count,
            complete_coverage,
        ],
    }
)

show_table(
    coverage_summary,
    "Cobertura global de valoraciones",
    formats={"Valor": "{:,.2f}"},
)

Indicador,Valor
Alojamientos en el último snapshot,"6,733.00"
Alojamientos con rating válido,"5,459.00"
Cobertura de rating,81.08
Alojamientos con cinco campos válidos,"5,443.00"
Cobertura completa de Experiencia,80.84


In [25]:
rating_coverage_city = coverage_by_group(
    dataframe=df_latest,
    group_column="city",
    value_column="review_scores_rating",
)

rating_coverage_city = rating_coverage_city.rename(
    columns={
        "city": "Ciudad","total": "Alojamientos","valid": "Rating válido","coverage_pct": "Cobertura",
    }
)

rating_coverage_city = rating_coverage_city.sort_values(
    "Cobertura",
    ascending=False,
)

show_table(
    rating_coverage_city,
    "Cobertura de rating por ciudad",
    formats={
        "Alojamientos": "{:,.0f}",
        "Rating válido": "{:,.0f}",
        "Cobertura": "{:.2f}%",
    },
)

Ciudad,Alojamientos,Rating válido,Cobertura
Valencia,297,279,93.94%
Sevilla,341,315,92.38%
Malaga,339,310,91.45%
Madrid,"1,396","1,194",85.53%
Menorca,138,116,84.06%
Barcelona,"2,041","1,697",83.15%
Mallorca,"1,096",794,72.45%
Girona,"1,085",754,69.49%


In [26]:
rating_coverage_room = coverage_by_group(
    dataframe=df_latest,
    group_column="room_type",
    value_column="review_scores_rating",
)

rating_coverage_room = rating_coverage_room.rename(
    columns={
        "room_type": "Tipo de alojamiento","total": "Alojamientos","valid": "Rating válido","coverage_pct": "Cobertura",
    }
)

rating_coverage_room = rating_coverage_room.sort_values(
    "Alojamientos",
    ascending=False,
)

show_table(
    rating_coverage_room,
    "Cobertura de rating por tipo de alojamiento",
    formats={
        "Alojamientos": "{:,.0f}",
        "Rating válido": "{:,.0f}",
        "Cobertura": "{:.2f}%",
    },
)

Tipo de alojamiento,Alojamientos,Rating válido,Cobertura
Entire home/apt,"4,780","3,869",80.94%
Private room,"1,875","1,526",81.39%
Shared room,43,31,72.09%
Hotel room,35,33,94.29%


### Interpretación

El **81,08 %** de los alojamientos tiene rating válido y el **80,84 %**
dispone de los cuatro componentes necesarios. Girona y Mallorca presentan la
menor cobertura. Los grupos pequeños de `room_type` requieren cautela.

## 8. Reserva instantánea y cobertura por ciudad

In [27]:
instant_by_city = pd.crosstab(
    df_latest["city"],
    df_latest["is_instant_bookable"],
)

instant_by_city = (
    instant_by_city
    .reset_index()
    .rename(columns={"city": "Ciudad"})
)

instant_columns = [
    column
    for column in ["FALSO", "VERDADERO"]
    if column in instant_by_city.columns
]

instant_by_city["Total"] = (
    instant_by_city[instant_columns]
    .sum(axis=1)
)

show_table(
    instant_by_city,
    "Reserva instantánea por ciudad",
    formats={
        column: "{:,.0f}"
        for column in instant_columns + ["Total"]
    },
)

Ciudad,FALSO,VERDADERO,Total
Barcelona,"1,112",929,"2,041"
Girona,453,632,"1,085"
Madrid,661,735,"1,396"
Malaga,110,229,339
Mallorca,460,636,"1,096"
Menorca,72,66,138
Sevilla,113,228,341
Valencia,162,135,297


### Interpretación

Las dos categorías de reserva instantánea aparecen en todas las ciudades.
La composición de los grupos varía territorialmente y debe controlarse antes
de comparar la disponibilidad media.

## 9. Formatos, escalas y consistencia semántica

In [28]:
range_columns = [
    "price","availability_30","availability_60","availability_90","availability_365","review_scores_rating",
    "review_scores_accuracy","review_scores_cleanliness","review_scores_checkin","review_scores_communication",
]

range_summary = pd.DataFrame(
    {
        "Campo": range_columns,
        "Mínimo": [
            df[column].min()
            for column in range_columns
        ],
        "Máximo": [
            df[column].max()
            for column in range_columns
        ],
        "Nulos": [
            df[column].isna().sum()
            for column in range_columns
        ],
    }
)

show_table(
    range_summary,
    "Rangos observados en variables principales",
    formats={
        "Mínimo": "{:,.2f}",
        "Máximo": "{:,.2f}",
        "Nulos": "{:,.0f}",
    },
)

Campo,Mínimo,Máximo,Nulos
price,60.00,"6,071.00",131
availability_30,0.00,30.00,0
availability_60,0.00,60.00,0
availability_90,0.00,90.00,0
availability_365,0.00,365.00,0
review_scores_rating,200.00,"1,000.00","1,327"
review_scores_accuracy,20.00,100.00,"1,336"
review_scores_cleanliness,20.00,100.00,"1,330"
review_scores_checkin,20.00,100.00,"1,341"
review_scores_communication,20.00,100.00,"1,332"


In [29]:
boolean_summary = pd.concat(
    {
        "is_instant_bookable": (
            df["is_instant_bookable"]
            .value_counts(dropna=False)
        ),
        "has_availability": (
            df["has_availability"]
            .value_counts(dropna=False)
        ),
    },
    axis=1,
)

boolean_summary = (
    boolean_summary
    .fillna(0)
    .astype(int)
    .rename_axis("Valor observado")
    .reset_index()
)

show_table(
    boolean_summary,
    "Valores observados en variables booleanas",
    formats={
        "is_instant_bookable": "{:,.0f}",
        "has_availability": "{:,.0f}",
    },
)

Valor observado,is_instant_bookable,has_availability
VERDADERO,"3,740","6,451"
FALSO,"3,261",0
—,0,550


In [30]:
date_columns = [
    "insert_date","first_review_date","last_review_date",
]

date_rows = []

for column in date_columns:
    parsed_dates = pd.to_datetime(
        df[column],
        format="%d/%m/%Y",
        errors="coerce",
    )

    date_rows.append(
        {
            "Campo": column,
            "Valores no nulos": df[column].notna().sum(),
            "Fechas interpretables": parsed_dates.notna().sum(),
            "Errores": (
                df[column].notna().sum()
                - parsed_dates.notna().sum()
            ),
            "Fecha mínima": parsed_dates.min(),
            "Fecha máxima": parsed_dates.max(),
        }
    )

date_summary = pd.DataFrame(date_rows)

show_table(
    date_summary,
    "Validación inicial de campos de fecha",
    formats={
        "Valores no nulos": "{:,.0f}",
        "Fechas interpretables": "{:,.0f}",
        "Errores": "{:,.0f}",
        "Fecha mínima": "{:%d/%m/%Y}",
        "Fecha máxima": "{:%d/%m/%Y}",
    },
)

Campo,Valores no nulos,Fechas interpretables,Errores,Fecha mínima,Fecha máxima
insert_date,"7,001","7,001",0,04/01/2017,27/02/2021
first_review_date,"5,747","5,747",0,02/01/2010,24/10/2020
last_review_date,"5,746","5,746",0,17/11/2012,13/02/2021


### Interpretación

`price` es numérico, pero presenta 131 nulos y un rango de 60 a 6.071 euros.
Las disponibilidades respetan sus rangos teóricos.

Las puntuaciones están almacenadas en una escala diez veces superior a la
documentada. La normalización corresponde a Data Transformation.

`is_instant_bookable` utiliza `VERDADERO/FALSO`. `has_availability` solo
contiene `VERDADERO` y nulos.

### 9.1 Estructura inicial de `amenities_list`

In [31]:
amenities = (
    df["amenities_list"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

translation_missing = (
    amenities
    .str.contains(
        "translation missing",
        case=False,
        na=False,
    )
    .sum()
)

amenities_summary = pd.DataFrame(
    {
        "Indicador": [
            "Filas con amenities_list nulo","Etiquetas distintas observadas","Apariciones de translation missing",
        ],
        "Valor": [
            df["amenities_list"].isna().sum(),
            amenities.nunique(),
            translation_missing,
        ],
    }
)

show_table(
    amenities_summary,
    "Estructura inicial de amenities_list",
    formats={"Valor": "{:,.0f}"},
)

Indicador,Valor
Filas con amenities_list nulo,17
Etiquetas distintas observadas,310
Apariciones de translation missing,"1,523"


In [32]:
text_columns = [
    "name","description","neighbourhood_name","neighbourhood_district","amenities_list",
]

encoding_rows = []

for column in text_columns:
    damaged_rows = (
        df[column]
        .astype("string")
        .str.contains(
            "\ufffd",
            regex=False,
            na=False,
        )
        .sum()
    )

    encoding_rows.append(
        {
            "Campo textual": column,
            "Filas con U+FFFD": damaged_rows,
        }
    )

encoding_summary = pd.DataFrame(encoding_rows)

show_table(
    encoding_summary,
    "Indicadores iniciales de codificación",
    formats={"Filas con U+FFFD": "{:,.0f}"},
)

Campo textual,Filas con U+FFFD
name,"1,391"
description,"4,769"
neighbourhood_name,"1,541"
neighbourhood_district,"1,003"
amenities_list,891


### Interpretación

`amenities_list` es un campo semiestructurado con 310 etiquetas distintas,
errores de traducción y caracteres dañados. Marketing necesitará una
transformación reproducible, pero esta fase no la ejecuta.

## 10. Población objetivo y subconjuntos analíticos

In [33]:
marketing_required = [
    "price","amenities_list","accommodates","bathrooms","bedrooms","beds","room_type","city","neighbourhood_name",
]

operations_required = [
    "is_instant_bookable","availability_30","availability_60","availability_90","availability_365","city",
]

experience_required = (
    experience_columns
    + [
        "apartment_id","city",
    ]
)

In [34]:
analytical_subsets = pd.DataFrame(
    {
        "Perfil": [
            "Marketing","Operaciones","Experiencia",
        ],
        "Población operativa": [
            len(df_latest),
            len(df_latest),
            len(df_latest),
        ],
        "Registros completos": [
            (
                df_latest[marketing_required]
                .notna()
                .all(axis=1)
                .sum()
            ),
            (
                df_latest[operations_required]
                .notna()
                .all(axis=1)
                .sum()
            ),
            (
                df_latest[experience_required]
                .notna()
                .all(axis=1)
                .sum()
            ),
        ],
        "Alcance": [
            "Precio y características válidas","Reserva y disponibilidad válidas","Rating y componentes válidos",
        ],
    }
)

show_table(
    analytical_subsets,
    "Subconjuntos analíticos por perfil",
    formats={
        "Población operativa": "{:,.0f}",
        "Registros completos": "{:,.0f}",
    },
)

Perfil,Población operativa,Registros completos,Alcance
Marketing,"6,733","6,539",Precio y características válidas
Operaciones,"6,733","6,733",Reserva y disponibilidad válidas
Experiencia,"6,733","5,443",Rating y componentes válidos


### Interpretación

La población operativa está formada por los **6.733 alojamientos** del último
snapshot disponible. Cada pregunta utiliza un subconjunto distinto. Las
conclusiones deben limitarse al subconjunto realmente analizado.

## 11. Posibles sesgos y limitaciones

En Data Understanding no se afirma que todos los sesgos estén demostrados.
Se documentan riesgos plausibles que podrían modificar sistemáticamente la
muestra, la medición o la interpretación.

| Riesgo | Perfil | Origen o impacto posible | Comprobación prevista |
|---|---|---|---|
| Selección y cobertura | Experiencia | Los alojamientos sin puntuación quedan fuera. | Comparar cobertura y limitar la inferencia. |
| Autoselección o no respuesta | Experiencia | La decisión de valorar puede relacionarse con experiencias extremas. | Documentar que no existen datos de quienes no reseñaron. |
| Representatividad desigual | Todos | Las ciudades y categorías tienen tamaños diferentes. | Reportar tamaños y evitar conclusiones con grupos pequeños. |
| Información y medición | Todos | Ratings subjetivos, texto dañado y disponibilidad imperfecta. | Normalizar, limpiar y definir cada variable. |
| Clasificación errónea | Marketing y Experiencia | El parsing o la definición Best/Worst puede cambiar resultados. | Validar reglas y comparar criterios. |
| Confusión | Todos | Ciudad, tipo, precio, ubicación, fecha y reseñas pueden influir. | Estratificar o ajustar en fases posteriores. |
| No contemporaneidad | Marketing y Operaciones | Los últimos snapshots pertenecen a años distintos. | Conservar `insert_date` y comprobar sensibilidad temporal. |
| Tamaño insuficiente | Todos | Grupos pequeños producen resultados inestables. | Definir mínimos y reportar tamaño del efecto. |

Referencia conceptual:
[Catalogue of Bias](https://catalogofbias.org/biases/).

### Alcance de la interpretación

Los riesgos identificados no demuestran automáticamente la existencia de
sesgo. Los resultados posteriores deberán expresarse como asociaciones,
diferencias o patrones observados, sin afirmar causalidad directa.

## 12. Adecuación del dataset para el Sprint 2

In [35]:
adequacy = pd.DataFrame(
    {
        "Perfil": [
            "Marketing y Estrategia Comercial",
            "Operaciones y Gestión de Inventario",
            "Experiencia del Cliente",
            "Conjunto del equipo",
        ],
        "Adecuación": [
            "Sí, con transformación adicional",
            "Sí, con cautela temporal",
            "Sí, para alojamientos evaluados",
            "Sí, dentro del alcance del ejercicio",
        ],
        "Conclusión": [
            "Contiene precio, equipamientos, capacidad y ubicación.",
            "Contiene reserva instantánea, disponibilidad y ciudad.",
            "Contiene rating general y los cuatro componentes.",
            "Debe generarse un dataset maestro por alojamiento.",
        ],
    }
)

show_table(
    adequacy,
    "Adecuación del dataset por perfil",
)

Perfil,Adecuación,Conclusión
Marketing y Estrategia Comercial,"Sí, con transformación adicional","Contiene precio, equipamientos, capacidad y ubicación."
Operaciones y Gestión de Inventario,"Sí, con cautela temporal","Contiene reserva instantánea, disponibilidad y ciudad."
Experiencia del Cliente,"Sí, para alojamientos evaluados",Contiene rating general y los cuatro componentes.
Conjunto del equipo,"Sí, dentro del alcance del ejercicio",Debe generarse un dataset maestro por alojamiento.


### Interpretación

No es necesaria otra fuente para abordar las preguntas dentro del alcance
**descriptivo y asociativo** del ejercicio. El dataset no permite demostrar
causalidad, medir reservas individuales ni representar precios actuales.

## 13. Decisiones metodológicas propuestas

| Decisión | Criterio propuesto |
|---|---|
| Unidad de análisis | Un alojamiento por `apartment_id`. |
| Snapshots | Conservar el registro más reciente según `insert_date`. |
| Precio | Investigar nulos, asimetría y outliers. |
| Equipamientos | Separar y limpiar `amenities_list`. |
| Capacidad | Validar rangos y evitar imputación automática. |
| Ubicación | Usar `city` como nivel común. |
| Reserva instantánea | Convertir `VERDADERO/FALSO` a booleano. |
| Disponibilidad | Comparar dentro de cada horizonte y normalizar entre horizontes. |
| Puntuaciones | Normalizar rating a 0-100 y componentes a 0-10. |
| Sin reseñas | Conservar en el maestro y excluir de satisfacción. |
| Best/Worst | Definir tras revisar la distribución. |
| Cobertura | Comparar el total con los subconjuntos. |
| Confusión | Evaluar ciudad, tipo, precio, ubicación, reseñas y fecha. |
| Temporalidad | Conservar `insert_date` y comprobar estabilidad. |
| Inferencia | Comunicar asociaciones y diferencias, no causalidad. |

La implementación definitiva corresponde a EDA, limpieza, transformación y
análisis.

## 14. Trazabilidad Notebook-PDF

In [36]:
expected_values = {
    "total_rows": 7001,
    "total_columns": 35,
    "unique_apartments": 6733,
    "additional_records": 268,
    "repeated_apartment_ids": 261,
    "exact_duplicates": 0,
    "cities": 8,
    "valid_ratings_latest": 5459,
    "complete_experience_latest": 5443,
    "neighbourhood_categories": 478,
    "amenity_labels": 310,
    "translation_missing": 1523,
}

actual_values = {
    "total_rows": len(df),
    "total_columns": df.shape[1],
    "unique_apartments": df["apartment_id"].nunique(),
    "additional_records": (
        len(df)
        - df["apartment_id"].nunique()
    ),
    "repeated_apartment_ids": (
        df["apartment_id"]
        .value_counts()
        .gt(1)
        .sum()
    ),
    "exact_duplicates": df.duplicated().sum(),
    "cities": df["city"].nunique(),
    "valid_ratings_latest": valid_rating_count,
    "complete_experience_latest": complete_experience_count,
    "neighbourhood_categories": (
        df_latest["neighbourhood_name"]
        .nunique()
    ),
    "amenity_labels": amenities.nunique(),
    "translation_missing": translation_missing,
}

In [37]:
validation_rows = []

for indicator, expected in expected_values.items():
    reproduced = actual_values[indicator]

    validation_rows.append(
        {
            "Indicador": indicator,
            "Valor esperado": expected,
            "Valor reproducido": reproduced,
            "Coincide": expected == reproduced,
        }
    )

validation = pd.DataFrame(validation_rows)

show_table(
    validation,
    "Validación de indicadores principales",
    formats={
        "Valor esperado": "{:,.0f}",
        "Valor reproducido": "{:,.0f}",
    },
)

if not validation["Coincide"].all():
    raise AssertionError(
        "Existen indicadores que no coinciden."
    )

print(
    "Validación completada: todos los "
    "indicadores principales coinciden."
)

Indicador,Valor esperado,Valor reproducido,Coincide
total_rows,"7,001","7,001",True
total_columns,35,35,True
unique_apartments,"6,733","6,733",True
additional_records,268,268,True
repeated_apartment_ids,261,261,True
exact_duplicates,0,0,True
cities,8,8,True
valid_ratings_latest,"5,459","5,459",True
complete_experience_latest,"5,443","5,443",True
neighbourhood_categories,478,478,True


Validación completada: todos los indicadores principales coinciden.


In [38]:
expected_city_counts = {
    "Barcelona": 2127,
    "Madrid": 1446,
    "Mallorca": 1144,
    "Girona": 1125,
    "Sevilla": 361,
    "Malaga": 350,
    "Valencia": 307,
    "Menorca": 141,
}

actual_city_counts = (
    df["city"]
    .value_counts()
    .to_dict()
)

if actual_city_counts != expected_city_counts:
    raise AssertionError(
        "Los recuentos por ciudad no coinciden."
    )

expected_year_counts = {
    2017: 773,
    2018: 1910,
    2019: 2151,
    2020: 1631,
    2021: 268,
}

actual_year_counts = (
    df_latest["insert_date_parsed"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_dict()
)

if actual_year_counts != expected_year_counts:
    raise AssertionError(
        "Los recuentos por año no coinciden."
    )

print("Validación ampliada completada.")

Validación ampliada completada.


## 15. Conclusión

El dataset contiene las variables necesarias para los tres análisis del
Sprint 2 dentro del alcance descriptivo y asociativo:

- **Marketing** deberá transformar equipamientos, capacidad y ubicación.
- **Operaciones** deberá conservar reserva instantánea, disponibilidad y fecha.
- **Experiencia del Cliente** deberá mantener rating, componentes y variables
  de control relacionadas con cobertura y posibles factores de confusión.

Se recomienda ampliar el pipeline existente para producir un dataset maestro
limpio con **6.733 alojamientos únicos**.

Las principales limitaciones son la codificación de `amenities_list`, la
cobertura territorial parcial, los alojamientos sin reseñas, la subjetividad
de las puntuaciones, la ausencia de transacciones individuales y la no
contemporaneidad de los últimos snapshots.

## Fuentes de referencia

- `Guió desafiament Nº2`.
- `Onboarding alojamientos`.
- `Instrucciones para Analistas de Datos Junior`.
- `raw_dataset_29_06_2026.csv`.
- `Data_Understanding_06_07_2026.pdf`.
- [Catalogue of Bias](https://catalogofbias.org/biases/).